# Grille 15 min de midquotes NBBO — univers S&P 500 PIT (WRDS TAQ)

Réplique la construction de données de Bollerslev, Li & Tang (Management Science 2026) :
rendements 15 min sur **midquotes NBBO**, 27 points par séance (09:30 → 16:00 ET), plus un rendement overnight.

Changements par rapport à la version précédente :
- clé **PERMNO** (lien WRDS CRSP-TAQM jour par jour, repli sur les tickers CRSP) au lieu du ticker seul ;
- point **09:30** ajouté (première NBBO valide de la séance) → 27 points, 26 rendements ;
- NBBO **propagée** sur les quinzaines sans changement (avant : ligne absente) ;
- filtre bid > 0 et ask ≥ bid **dans la requête** (dernière cotation *valide*, et non dernière cotation invalidée) ;
- un jour en erreur n'annule plus tout le mois ; fichier de couverture par mois ;
- rendement overnight, avec reprise d'un mois sur l'autre.

Aucun filtre de nom n'est appliqué (share code 10/11, prix 1–1000 $, ≥ 100 transactions/jour).

In [ ]:
# 1. Univers PIT (sortie de build_sp500_pit_ids.py). Aucun filtre de nom appliqué ici.
#    Filtres de l'article (documentés, NON appliqués) : NYSE/NASDAQ/AMEX, share code 10/11,
#    prix 1–1000 $, >= 100 transactions/jour, historique complet 2000–2020.
import pandas as pd
import numpy as np
from pathlib import Path

SEGMENTS = "sp500_pit_identifiers_segments.csv"   # CSV de préférence : garde les zéros de gvkey/CUSIP
DATE_COLS = ["valid_from", "valid_to", "member_from", "member_to"]
STR_COLS = {"gvkey": str, "cusip8": str, "cusip9": str, "crsp_ticker": str,
            "crsp_tradingsymbol": str, "shareclass": str}


def load_segments(path):
    path = Path(path)
    if path.suffix in (".xlsx", ".xls"):
        seg = pd.read_excel(path, dtype=STR_COLS)
    else:
        seg = pd.read_csv(path, dtype=STR_COLS)
    for col in DATE_COLS:
        if pd.api.types.is_numeric_dtype(seg[col]):           # dates sérialisées par Excel
            seg[col] = pd.to_datetime(seg[col], unit="D", origin="1899-12-30")
        else:
            seg[col] = pd.to_datetime(seg[col])
    return seg


df = load_segments(SEGMENTS)
print(len(df), "segments,", df["permno"].nunique(), "PERMNO")

In [ ]:
# 2. Univers d'un mois : tout PERMNO membre à un moment du mois (on télécharge le mois entier ;
#    l'appartenance se réapplique à la date d'événement en aval).
def get_monthly_universe(month):
    month = pd.Period(month, freq="M")
    start, end = month.start_time.normalize(), month.end_time.normalize()
    mask = ((df["valid_from"] <= end) & (df["valid_to"] >= start)
            & (df["member_from"] <= end) & (df["member_to"] >= start))
    return df.loc[mask].copy()


def crsp_to_taq_symbol(ticker, tradingsymbol, shareclass):
    """Repli sans table de lien : CRSP (BRK, BRKB, B) -> TAQ (sym_root=BRK, sym_suffix=B)."""
    if (isinstance(shareclass, str) and isinstance(tradingsymbol, str)
            and isinstance(ticker, str) and tradingsymbol == f"{ticker}{shareclass}"):
        return ticker, shareclass
    sym = tradingsymbol if isinstance(tradingsymbol, str) else ticker
    return sym, ""

In [ ]:
%pip install wrds pandas_market_calendars pyarrow tqdm

In [ ]:
# 3. Connexion WRDS et vérification des schémas (à lancer une fois, lire la sortie).
import wrds
import pandas_market_calendars as mcal
from tqdm.auto import tqdm

conn = wrds.Connection(wrds_username="ethan099")

print(conn.list_tables("wrdsapps_link_crsp_taqm"))
# Colonnes de la table de lien PERMNO <-> symbole TAQ (adapter LINK_* ci-dessous si besoin)
print(conn.describe_table("wrdsapps_link_crsp_taqm", "taqmclink"))
# Colonnes d'une table NBBO quotidienne
print(conn.describe_table("taqm_2008", "complete_nbbo_20080102"))

In [ ]:
# 4. Correspondance PERMNO -> (sym_root, sym_suffix), jour par jour.
#    Source principale : table de lien WRDS CRSP-TAQM (PIT). Repli : tickers CRSP du segment.
LINK_TABLE = "wrdsapps_link_crsp_taqm.taqmclink"
LINK_DATE, LINK_PERMNO, LINK_ROOT, LINK_SUFFIX = "date", "permno", "sym_root", "sym_suffix"
LINK_MATCH = "match_lvl"          # None si la colonne n'existe pas
USE_LINK = True


def get_symbol_map(month, universe, trading_days):
    """DataFrame [date, permno, sym_root, sym_suffix, map_source] pour les jours du mois."""
    permnos = tuple(int(p) for p in universe["permno"].unique())
    start, end = min(trading_days), max(trading_days)
    link = pd.DataFrame()
    if USE_LINK:
        cols = f"{LINK_DATE} AS date, {LINK_PERMNO} AS permno, {LINK_ROOT} AS sym_root, {LINK_SUFFIX} AS sym_suffix"
        if LINK_MATCH:
            cols += f", {LINK_MATCH} AS match_lvl"
        try:
            link = conn.raw_sql(
                f"SELECT {cols} FROM {LINK_TABLE} "
                f"WHERE {LINK_DATE} BETWEEN %(s)s AND %(e)s AND {LINK_PERMNO} IN %(p)s",
                params={"s": start.date(), "e": end.date(), "p": permnos}, date_cols=["date"])
        except Exception as e:
            tqdm.write(f"lien CRSP-TAQ indisponible ({e}) -> repli tickers CRSP")
            link = pd.DataFrame()
    if not link.empty:
        link["sym_root"] = link["sym_root"].str.strip()
        link["sym_suffix"] = link["sym_suffix"].fillna("").str.strip()
        if "match_lvl" in link:
            link = link.sort_values("match_lvl")
        link = link.drop_duplicates(["date", "permno"]).assign(map_source="link")

    # repli pour les (jour, PERMNO) absents du lien
    rows = []
    for r in universe.itertuples():
        root, suffix = crsp_to_taq_symbol(r.crsp_ticker, r.crsp_tradingsymbol, r.shareclass)
        for d in trading_days:
            if r.valid_from <= d <= r.valid_to:
                rows.append((d, int(r.permno), root, suffix))
    fb = pd.DataFrame(rows, columns=["date", "permno", "sym_root", "sym_suffix"]).assign(map_source="crsp")
    out = pd.concat([link[fb.columns] if not link.empty else fb.iloc[:0], fb], ignore_index=True)
    out["date"] = pd.to_datetime(out["date"]).dt.normalize()
    out = out[out["date"].isin(trading_days)]
    return out.drop_duplicates(["date", "permno"], keep="first").reset_index(drop=True)

In [ ]:
# 5. Grille 15 min de midquotes NBBO pour un jour.
#    Points : 09:30, 09:45, ..., clôture (16:00 ou 13:00 les demi-séances) -> 27 points / jour normal.
#    09:30 = première cotation NBBO valide à partir de l'ouverture (fenêtre OPEN_WINDOW).
#    t > 09:30 = dernière NBBO valide strictement avant t, propagée si la NBBO n'a pas changé.
FREQ = "15min"
OPEN_WINDOW = "00:05:00"      # cherche la cotation d'ouverture dans [09:30, 09:35)
BATCH_SIZE = 250              # symboles par requête SQL


def _nbbo_queries(table, roots, seconds, market_close):
    valid = "best_bid > 0 AND best_ask >= best_bid"
    last_q = f"""
        WITH q AS (
            SELECT sym_root, sym_suffix, time_m, best_bid, best_ask,
                   (FLOOR(EXTRACT(EPOCH FROM time_m) / %(seconds)s) + 1) * %(seconds)s AS bucket_end
            FROM {table}
            WHERE sym_root IN %(roots)s
              AND time_m >= '09:30:00' AND time_m < %(close)s AND {valid}
        )
        SELECT DISTINCT ON (sym_root, sym_suffix, bucket_end)
               sym_root, sym_suffix, bucket_end, time_m AS quote_time,
               best_bid AS bid, best_ask AS ask
        FROM q
        ORDER BY sym_root, sym_suffix, bucket_end, time_m DESC
    """
    open_q = f"""
        SELECT DISTINCT ON (sym_root, sym_suffix)
               sym_root, sym_suffix, time_m AS quote_time, best_bid AS bid, best_ask AS ask
        FROM {table}
        WHERE sym_root IN %(roots)s
          AND time_m >= '09:30:00' AND time_m < %(open_end)s AND {valid}
        ORDER BY sym_root, sym_suffix, time_m ASC
    """
    params = {"roots": roots, "seconds": seconds, "close": market_close,
              "open_end": str(pd.Timedelta("09:30:00") + pd.Timedelta(OPEN_WINDOW))[-8:]}
    return last_q, open_q, params


def build_day_grid(date, symmap_day, last, opn, freq, market_close):
    """Assemble la grille d'un jour à partir des deux résultats SQL (testable hors WRDS)."""
    date = pd.Timestamp(date).normalize()
    close_ts = date + pd.Timedelta(market_close)
    grid_times = pd.date_range(date + pd.Timedelta("09:30:00"), close_ts, freq=freq)
    for d in (last, opn):
        d["sym_root"] = d["sym_root"].str.strip()
        d["sym_suffix"] = d["sym_suffix"].fillna("").str.strip()
    last = last.assign(grid_time=date + pd.to_timedelta(last["bucket_end"].astype(float), unit="s"))
    opn = opn.assign(grid_time=grid_times[0])
    q = pd.concat([opn, last.drop(columns="bucket_end")], ignore_index=True)
    q["quote_time"] = date + pd.to_timedelta(q["quote_time"].astype(str))

    sm = symmap_day[["permno", "sym_root", "sym_suffix", "map_source"]]
    q = q.merge(sm, on=["sym_root", "sym_suffix"], how="inner")
    full = (sm.assign(_k=1).merge(pd.DataFrame({"grid_time": grid_times, "_k": 1}), on="_k").drop(columns="_k"))
    full = full.merge(q[["permno", "grid_time", "quote_time", "bid", "ask"]], on=["permno", "grid_time"], how="left")
    full = full.sort_values(["permno", "grid_time"])
    # la NBBO reste valable tant qu'elle ne change pas : propagation dans la journée
    full[["quote_time", "bid", "ask"]] = full.groupby("permno")[["quote_time", "bid", "ask"]].ffill()
    full["date"] = date
    full["stale_sec"] = (full["grid_time"] - full["quote_time"]).dt.total_seconds()
    full["mid"] = (full["bid"] + full["ask"]) / 2
    full["spread"] = full["ask"] - full["bid"]
    full["ret_15m"] = np.log(full["mid"]).groupby(full["permno"]).diff()
    full["grid_time"] = full["grid_time"].dt.tz_localize("America/New_York")
    full["quote_time"] = full["quote_time"].dt.tz_localize("America/New_York")
    return full.reset_index(drop=True)


def get_nbbo_day(date, symmap_day, freq=FREQ, market_close="16:00:00"):
    date = pd.Timestamp(date)
    table = f"taqm_{date.year}.complete_nbbo_{date:%Y%m%d}"
    seconds = int(pd.Timedelta(freq).total_seconds())
    roots_all = sorted(symmap_day["sym_root"].unique())
    lasts, opens = [], []
    for i in range(0, len(roots_all), BATCH_SIZE):
        roots = tuple(roots_all[i:i + BATCH_SIZE])
        last_q, open_q, params = _nbbo_queries(table, roots, seconds, market_close)
        params["roots"] = roots
        lasts.append(conn.raw_sql(last_q, params=params))
        opens.append(conn.raw_sql(open_q, params=params))
    return build_day_grid(date, symmap_day, pd.concat(lasts, ignore_index=True),
                          pd.concat(opens, ignore_index=True), freq, market_close)

In [ ]:
# 6. Téléchargement mensuel (reprenable). Un parquet par mois + un fichier de couverture.
#    TAQ millisecondes (taqm_*) : vérifier le premier jour disponible de taqm_2003 avant de
#    descendre START sous 2008 ; avant, il faudrait la TAQ mensuelle (non listée ici).
START, END = "2008-01", "2026-06"
OUTPUT_DIR = Path(f"data_wrds_{FREQ}")
OUTPUT_DIR.mkdir(exist_ok=True)

nyse = mcal.get_calendar("NYSE")
months = pd.period_range(START, END, freq="M")
failed = {}
pbar = tqdm(months, desc="Mois", unit="mois")

for month in pbar:
    month_str = str(month)
    filepath = OUTPUT_DIR / f"{month_str}.parquet"
    if filepath.exists():
        continue
    try:
        universe = get_monthly_universe(month_str)
        schedule = nyse.schedule(start_date=month.start_time.date(), end_date=month.end_time.date())
        days = pd.DatetimeIndex(schedule.index).normalize()
        symmap = get_symbol_map(month_str, universe, days)

        monthly, coverage = [], []
        for date, session in tqdm(schedule.iterrows(), total=len(schedule),
                                  desc=f"{month_str} {universe['permno'].nunique()} PERMNO", leave=False):
            d = pd.Timestamp(date).normalize()
            market_close = session["market_close"].tz_convert("America/New_York").strftime("%H:%M:%S")
            sm = symmap[symmap["date"] == d]
            try:
                day = get_nbbo_day(d, sm, FREQ, market_close)
            except Exception as e:
                failed[str(d.date())] = str(e)[:200]
                tqdm.write(f"ERREUR {d.date()}: {e}")
                continue
            got = day.dropna(subset=["mid"])["permno"].nunique()
            coverage.append((d.date(), len(sm), got, (sm["map_source"] == "crsp").sum()))
            monthly.append(day)

        if not monthly:
            raise ValueError("aucun jour récupéré")
        out = pd.concat(monthly, ignore_index=True)
        tmp = filepath.with_suffix(".tmp.parquet")
        out.to_parquet(tmp, index=False, compression="snappy")
        tmp.replace(filepath)
        pd.DataFrame(coverage, columns=["date", "permno_expected", "permno_with_mid", "n_crsp_fallback"]) \
          .to_csv(OUTPUT_DIR / f"{month_str}_coverage.csv", index=False)
        pbar.set_postfix(saved=month_str, rows=f"{len(out):,}", errors=len(failed))
    except Exception as e:
        failed[month_str] = str(e)[:200]
        tqdm.write(f"ERREUR {month_str}: {e}")

print("TERMINÉ ; échecs :", failed)

In [ ]:
# 7. Rendement overnight (clôture 16:00/13:00 de la veille -> 09:30), mois par mois en
#    reportant la dernière clôture d'un mois sur l'autre. Brut : NON ajusté des splits/dividendes.
#    Les jours de split (facteur CRSP non nul) sont mis à NaN si CRSP_DSF_DIR est renseigné.
CRSP_DSF_DIR = None        # ex. "crsp_dsf_v2" contenant crsp_dsf_v2_YYYY.parquet (Drive)
CRSP_FAC_COL = "dlyfacprc" # facteur d'ajustement de prix CRSP CIZ (vérifier le nom de colonne)


def load_split_days():
    if not CRSP_DSF_DIR:
        return None
    parts = []
    for f in sorted(Path(CRSP_DSF_DIR).glob("crsp_dsf_v2_*.parquet")):
        d = pd.read_parquet(f, columns=["permno", "dlycaldt", CRSP_FAC_COL])
        parts.append(d[d[CRSP_FAC_COL].fillna(0) != 0])
    s = pd.concat(parts)
    return set(zip(s["permno"].astype(int), pd.to_datetime(s["dlycaldt"]).dt.normalize()))


def add_overnight(month_df, prev_close, split_days=None):
    """prev_close : {permno: log mid de clôture de la dernière séance du mois précédent}.
    Un PERMNO absent la veille -> NaN (pas de saut sur plusieurs séances)."""
    m = month_df.sort_values(["permno", "grid_time"]).copy()
    m["logmid"] = np.log(m["mid"])
    days = sorted(m["date"].unique())
    first = m.groupby(["permno", "date"]).head(1).set_index(["permno", "date"])["logmid"]
    last = m.groupby(["permno", "date"]).tail(1).set_index(["permno", "date"])["logmid"]
    on = {}
    for i, d in enumerate(days):
        for p, lo in first.xs(d, level="date").items():
            pc = prev_close.get(p) if i == 0 else last.get((p, days[i - 1]))
            on[(p, d)] = lo - pc if pc is not None else np.nan
    is_open = m.groupby(["permno", "date"]).cumcount() == 0
    key = list(zip(m["permno"], m["date"]))
    m["ret_overnight_raw"] = np.where(is_open, [on.get(k, np.nan) for k in key], np.nan)
    if split_days:
        hit = np.array([k in split_days for k in key])
        m.loc[is_open & hit, "ret_overnight_raw"] = np.nan
    new_close = last.xs(days[-1], level="date").to_dict()
    return m.drop(columns="logmid"), new_close


split_days = load_split_days()
prev_close = {}
FINAL_DIR = OUTPUT_DIR / "with_overnight"
FINAL_DIR.mkdir(exist_ok=True)
for f in sorted(OUTPUT_DIR.glob("????-??.parquet")):
    m = pd.read_parquet(f)
    m["date"] = pd.to_datetime(m["date"]).dt.normalize()
    m, prev_close = add_overnight(m, prev_close, split_days)
    m.to_parquet(FINAL_DIR / f.name, index=False)
    print(f.name, f"{m['ret_overnight_raw'].notna().sum():,} rendements overnight")